# 02 · Removing the leaks

Notebook 01 scored about 99.9%, but the data gave the answer away in two ways:

1. **Scheme leak.** 100% of legit URLs start with `https://www.`, versus 2.4% of phishing. `has_https` was the top feature.
2. **Path leak.** Legit URLs are bare homepages (max length 58), so *any* path screams phishing. `path_len` was the #2 feature and the biggest logistic regression weight.

Plan: rerun the same model on progressively cleaner inputs and watch the score fall to something honest. Then try a stronger model (character n-grams), and finally test on realistic legit URLs the dataset never had.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

from phishdetect.data import load_phiusiil
from phishdetect.features import featurize, normalize_url, host_only, FEATURE_NAMES

pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

## 1. Load and split once
We split row *indices*, so every experiment below trains and tests on exactly the same URLs. That keeps the comparison fair.

In [ ]:
df = load_phiusiil().drop_duplicates("url").reset_index(drop=True)
y = df["is_phishing"].values
idx_train, idx_test = train_test_split(
    np.arange(len(df)), test_size=0.2, stratify=y, random_state=RANDOM_STATE)
y_train, y_test = y[idx_train], y[idx_test]
print(len(idx_train), len(idx_test))

## 2. Three views of the same URL

In [ ]:
df["normalized"] = df["url"].map(normalize_url)   # scheme and www. removed
df["host"] = df["url"].map(host_only)             # hostname only
df.sample(8, random_state=1)[["url", "normalized", "host", "is_phishing"]]

Did stripping the prefix fix the scheme leak? Check what share of each class still has a path after normalizing:

In [ ]:
has_path = df["normalized"].str.contains("/", regex=False) & ~df["normalized"].str.endswith("/")
pd.crosstab(has_path.rename("has_path"), df["is_phishing"].map({0: "legit", 1: "phishing"}), normalize="columns").round(3)

If legit is close to 0% with a path, the path leak is still there in the normalized view. Only the host-only view fixes it, because then neither class has a path.

## 3. Same model, cleaner inputs

| experiment | input | features dropped |
|---|---|---|
| A · original | raw URL | none (same as notebook 01) |
| B · no scheme | normalized URL | `has_https` |
| C · host only | hostname | `has_https`, and all path/query features |

In [ ]:
PATH_FEATURES = ["path_len", "query_len", "has_double_slash_path"]

experiments = {
    "A · original":  (df["url"],        []),
    "B · no scheme": (df["normalized"], ["has_https"]),
    "C · host only": (df["host"],       ["has_https"] + PATH_FEATURES),
}

def scores(y_true, proba):
    pred = (proba >= 0.5).astype(int)
    return {"precision": precision_score(y_true, pred), "recall": recall_score(y_true, pred),
            "f1": f1_score(y_true, pred), "PR-AUC": average_precision_score(y_true, proba)}

rows, fitted = [], {}
for name, (text, drop) in experiments.items():
    X = featurize(text).drop(columns=drop)
    rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE)
    rf.fit(X.iloc[idx_train], y_train)
    proba = rf.predict_proba(X.iloc[idx_test])[:, 1]
    rows.append({"experiment": name, **scores(y_test, proba)})
    fitted[name] = (rf, list(X.columns))
    print("done", name)

results = pd.DataFrame(rows).set_index("experiment").round(4)
results

The drop from A to C is how much of notebook 01's score came from collection artifacts rather than anything about phishing. **C is the honest baseline.**

In [ ]:
rf_c, cols_c = fitted["C · host only"]
pd.Series(rf_c.feature_importances_, index=cols_c).sort_values().plot.barh(
    figsize=(7, 5), title="Host-only random forest: feature importance"); plt.show()

## 4. A stronger model: character n-grams

Hand-written features only capture what we thought of. A character n-gram model breaks each hostname into overlapping chunks (`pay`, `ayp`, `ypa`, `pal`, ... for 3-grams) and learns which chunks show up in phishing. It picks up brand names, odd letter combos, and TLDs on its own.

- `TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))` turns each hostname into a sparse vector of 3- to 5-character chunks, weighted so common chunks count less.
- `min_df=3` ignores chunks seen in fewer than 3 hostnames (noise).
- Logistic regression on top works well with large sparse inputs and is fast.

In [ ]:
%%time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

char_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, sublinear_tf=True, max_features=200_000),
    LogisticRegression(max_iter=2000, C=4.0),
)
hosts = df["host"].values
char_model.fit(hosts[idx_train], y_train)
proba_char = char_model.predict_proba(hosts[idx_test])[:, 1]

results.loc["D · host char n-grams"] = pd.Series(scores(y_test, proba_char)).round(4)
results

What did it learn? The n-grams with the biggest weights in each direction:

In [ ]:
vec, clf = char_model[0], char_model[-1]
weights = pd.Series(clf.coef_[0], index=vec.get_feature_names_out())
print("Most phishy:", list(weights.nlargest(25).index))
print()
print("Most legit: ", list(weights.nsmallest(25).index))

## 5. Reality check: URLs the dataset never had

The training set's legit URLs are all bare homepages. Real traffic isn't. Score some everyday legit links with paths, plus a few made-up phishing-style URLs, and see what each model does.

- **B** sees the full path, so if it learned "has a path means phishing" it will flag the real links.
- **C** and **D** only see the hostname, so they can't fall for that.

In [ ]:
legit_real = [
    "https://www.amazon.com/dp/B0CHX1W1XY?ref=ppx_yo2ov_dt_b_product_details",
    "https://github.com/scikit-learn/scikit-learn/blob/main/README.rst",
    "https://docs.google.com/document/d/1aBcD3fGhIjKlMnOpQrStUvWxYz/edit",
    "https://www.vcu.edu/admissions/graduate/",
    "https://en.wikipedia.org/wiki/Phishing",
    "https://mail.google.com/mail/u/0/#inbox",
    "https://www.linkedin.com/jobs/view/1234567890/",
    "https://stackoverflow.com/questions/231767/what-does-the-yield-keyword-do",
]
phish_style = [
    "http://paypal-account-verify.xyz/login",
    "https://secure-login-appleid.top/signin?session=8f3a",
    "http://chase-online-banking.co/update/billing",
    "https://microsoft365-support-verify.app/",
]
check = pd.DataFrame({"url": legit_real + phish_style,
                      "truth": ["legit"] * len(legit_real) + ["phish-style"] * len(phish_style)})

for name, view in [("B · no scheme", normalize_url), ("C · host only", host_only)]:
    model, cols = fitted[name]
    Xc = featurize(check["url"].map(view))[cols]
    check[f"p_phish {name[0]}"] = model.predict_proba(Xc)[:, 1].round(3)
check["p_phish D"] = char_model.predict_proba(check["url"].map(host_only))[:, 1].round(3)
check

Read the B column for the legit rows. High scores there are the path leak in action: a model that looks great on the test set but would flag a huge share of real links. Compare C and D.

Note this is 12 hand-picked URLs, not a real evaluation. It shows the failure mode; it doesn't measure it. A proper fix is adding legit URLs *with paths* to the training data, which is a good task for later.

## 6. Save the honest model

In [ ]:
import joblib
from phishdetect.data import PROJECT_ROOT
best_name = results.loc[["C · host only", "D · host char n-grams"], "PR-AUC"].idxmax()
best = char_model if best_name.startswith("D") else fitted["C · host only"][0]
joblib.dump(best, PROJECT_ROOT / "models" / "host_model.joblib")
(PROJECT_ROOT / "models" / "host_model_metrics.json").write_text(results.loc[best_name].to_json(indent=2))
print("saved", best_name)

## Results log

Fill in after running, and copy to the README.

| experiment | precision | recall | PR-AUC |
|---|---|---|---|
| A · original | | | |
| B · no scheme | | | |
| C · host only | | | |
| D · host char n-grams | | | |

**Talking point:** "The published-style score was 99.8%. I found the dataset leaked the label through the URL scheme and through legit URLs being homepage-only. After restricting to hostnames, the honest score is ___, and a character n-gram model brings it to ___."